# Part 2 - N-gram Language Models

Corpus: NLTK Gutenberg, 80% train / 20% test at sentence level with a fixed
seed. Any train word with count = 1 collapses to `<UNK>`, so unigram, bigram
and trigram models share one vocabulary. This notebook is self-contained; every
helper it needs is defined below.

Coverage: Q6 MLE unigram/bigram, Q7 add-1 (Laplace) smoothing, Q8 perplexity
table, Q9 five ancestral samples from the bigram, Q10 trigram and comparison.

## Setup

In [1]:
from __future__ import annotations

import json
import logging
import math
import random
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

In [2]:
_REQUIRED_NLTK = (
    ("corpora/gutenberg", "gutenberg"),
    ("corpora/brown", "brown"),
    ("taggers/universal_tagset", "universal_tagset"),
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
)


def ensure_corpora() -> None:
    for resource_path, pkg in _REQUIRED_NLTK:
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(pkg, quiet=True)


ensure_corpora()

In [3]:
@dataclass(frozen=True)
class Part2Config:
    corpus_name: str = "gutenberg"
    seed: int = 42
    lowercase: bool = True
    keep_only_alpha: bool = True
    train_ratio: float = 0.8
    unk_min_count: int = 1
    ngram_orders: Tuple[int, ...] = (1, 2, 3)
    laplace_alpha: float = 1.0
    num_samples: int = 5
    max_sample_length: int = 40
    log_file: str = "part2_lm.log"


cfg = Part2Config()
LOG_DIR = REPO / "logs"
ART_DIR = REPO / "artifacts" / "part2_lm"
OUT_DIR = REPO / "outputs"
for d in (LOG_DIR, ART_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [4]:
_LOG_FORMAT = "%(asctime)s | %(levelname)-5s | %(name)s | %(message)s"
_DATE_FORMAT = "%Y-%m-%d %H:%M:%S"


def get_logger(name: str, log_file: Path | str, level: str | int = "INFO") -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = False
    for h in list(logger.handlers):
        logger.removeHandler(h)
        h.close()
    log_path = Path(log_file)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    formatter = logging.Formatter(_LOG_FORMAT, _DATE_FORMAT)
    fh = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)
    sh = logging.StreamHandler(stream=sys.stdout)
    sh.setFormatter(formatter)
    logger.addHandler(sh)
    return logger


def _format_metrics(metrics: Mapping[str, Any]) -> str:
    parts = []
    for k, v in metrics.items():
        parts.append(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}")
    return " ".join(parts)


def log_iteration(logger: logging.Logger, step: int, total: int | None,
                  prefix: str = "iter", **metrics: Any) -> None:
    header = f"{prefix} {step}/{total}" if total is not None else f"{prefix} {step}"
    body = _format_metrics(metrics)
    logger.info(f"{header} {body}".rstrip())


def log_section(logger: logging.Logger, title: str) -> None:
    logger.info(f"--- {title} ---")

In [5]:
def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, ensure_ascii=False)


def save_text(lines: Iterable[str], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for line in lines:
            fh.write(line.rstrip("\n") + "\n")

In [6]:
def split_sentences(sentences: Sequence, train_ratio: float, seed: int) -> Tuple[List, List]:
    rng = random.Random(seed)
    idx = list(range(len(sentences)))
    rng.shuffle(idx)
    cut = int(len(idx) * train_ratio)
    train = [sentences[i] for i in idx[:cut]]
    test = [sentences[i] for i in idx[cut:]]
    return train, test

## Corpus loader and n-gram language model

In [7]:
def _is_token_kept(token: str, keep_only_alpha: bool) -> bool:
    if not keep_only_alpha:
        return True
    return any(c.isalpha() for c in token)


def load_sentences(corpus_name: str, lowercase: bool = True,
                   keep_only_alpha: bool = True) -> List[List[str]]:
    if corpus_name == "gutenberg":
        raw_sents = nltk.corpus.gutenberg.sents()
    elif corpus_name == "brown":
        raw_sents = nltk.corpus.brown.sents()
    else:
        raise ValueError(f"unknown corpus: {corpus_name!r}")
    sents: List[List[str]] = []
    for s in raw_sents:
        toks = [t.lower() if lowercase else t for t in s]
        toks = [t for t in toks if _is_token_kept(t, keep_only_alpha)]
        if toks:
            sents.append(toks)
    return sents

In [8]:
BOS, EOS, UNK = "<s>", "</s>", "<UNK>"


def _replace_oov(sentences: Sequence[Sequence[str]], vocab: set) -> List[List[str]]:
    return [[t if t in vocab else UNK for t in s] for s in sentences]


def build_vocab(train_sentences: Sequence[Sequence[str]], unk_min_count: int) -> set:
    counts: Counter = Counter()
    for s in train_sentences:
        counts.update(s)
    vocab = {t for t, c in counts.items() if c > unk_min_count}
    vocab.update({UNK, BOS, EOS})
    return vocab


class NgramLanguageModel:
    def __init__(self, order: int, alpha: float = 1.0):
        if order not in (1, 2, 3):
            raise ValueError("order must be 1, 2 or 3")
        self.order = order
        self.alpha = alpha
        self.vocab: set = set()
        self._ctx_counts: Dict[Tuple[str, ...], Counter] = defaultdict(Counter)
        self._ctx_totals: Dict[Tuple[str, ...], int] = defaultdict(int)
        self._V = 0

    def _pad(self, sentence: Sequence[str]) -> List[str]:
        return [BOS] * (self.order - 1) + list(sentence) + [EOS]

    def fit(self, train_sentences: Sequence[Sequence[str]], vocab: set,
            logger: logging.Logger | None = None) -> None:
        self.vocab = set(vocab)
        self._V = len(self.vocab)
        self._ctx_counts.clear()
        self._ctx_totals.clear()
        start = time.time()
        for s in _replace_oov(train_sentences, self.vocab):
            padded = self._pad(s)
            for i in range(self.order - 1, len(padded)):
                ctx = tuple(padded[i - self.order + 1 : i])
                self._ctx_counts[ctx][padded[i]] += 1
                self._ctx_totals[ctx] += 1
        if logger is not None:
            unique = sum(len(c) for c in self._ctx_counts.values())
            log_iteration(logger, self.order, 3, prefix="fit",
                          order=self.order, vocab=self._V,
                          contexts=len(self._ctx_counts),
                          unique_ngrams=unique,
                          elapsed_s=time.time() - start)

    def prob(self, context: Tuple[str, ...], word: str, smoothed: bool = True) -> float:
        c_ctx_w = self._ctx_counts[context].get(word, 0)
        c_ctx = self._ctx_totals.get(context, 0)
        if smoothed:
            return (c_ctx_w + self.alpha) / (c_ctx + self.alpha * self._V)
        if c_ctx == 0:
            return 0.0
        return c_ctx_w / c_ctx

    def log_prob_token(self, context: Tuple[str, ...], word: str) -> float:
        c_ctx_w = self._ctx_counts[context].get(word, 0)
        c_ctx = self._ctx_totals.get(context, 0)
        num = c_ctx_w + self.alpha
        den = c_ctx + self.alpha * self._V
        return math.log(num) - math.log(den)

    def log_prob_sentence(self, sentence: Sequence[str]) -> Tuple[float, int]:
        padded = self._pad(sentence)
        total, n = 0.0, 0
        for i in range(self.order - 1, len(padded)):
            ctx = tuple(padded[i - self.order + 1 : i])
            total += self.log_prob_token(ctx, padded[i])
            n += 1
        return total, n

    def perplexity(self, test_sentences: Sequence[Sequence[str]],
                   logger: logging.Logger | None = None) -> Tuple[float, int, float]:
        mapped = _replace_oov(test_sentences, self.vocab)
        total_lp, total_n = 0.0, 0
        for s in mapped:
            lp, n = self.log_prob_sentence(s)
            total_lp += lp
            total_n += n
        avg_neg = -total_lp / max(1, total_n)
        ppl = math.exp(avg_neg)
        if logger is not None:
            log_iteration(logger, self.order, 3, prefix="ppl",
                          order=self.order, tokens_scored=total_n,
                          log_prob_sum=total_lp, ppl=ppl)
        return ppl, total_n, total_lp

    def sample(self, rng: np.random.Generator, max_length: int,
               use_smoothing: bool = False) -> List[str]:
        if self.order == 1:
            raise ValueError("sampling from a unigram model is degenerate")
        context: Tuple[str, ...] = tuple([BOS] * (self.order - 1))
        out: List[str] = []
        for _ in range(max_length):
            nxt = self._draw_next(context, rng, use_smoothing)
            if nxt == EOS:
                break
            if nxt == BOS:
                continue
            out.append(nxt)
            context = tuple((list(context) + [nxt])[-(self.order - 1) :])
        return out

    def _draw_next(self, context: Tuple[str, ...], rng: np.random.Generator,
                   use_smoothing: bool) -> str:
        counts = self._ctx_counts[context]
        total = self._ctx_totals.get(context, 0)
        if not use_smoothing and total > 0:
            words = list(counts.keys())
            probs = np.asarray([counts[w] for w in words], dtype=np.float64)
            probs /= probs.sum()
            return str(rng.choice(words, p=probs))
        vocab_list = sorted(self.vocab)
        probs = np.full(len(vocab_list), self.alpha, dtype=np.float64)
        for i, w in enumerate(vocab_list):
            probs[i] += counts.get(w, 0)
        probs /= probs.sum()
        return vocab_list[int(rng.choice(len(vocab_list), p=probs))]

## Shared training and evaluation pipeline

Every Q section below reads from the two objects built here: `models` (a dict
of order -> `NgramLanguageModel`) and `report` (a list of dicts with the
perplexity of each order on the same held-out test set). Nothing is saved to
disk in this cell; the Q sections write their own artefacts.

In [9]:
logger = get_logger("part2_lm", LOG_DIR / cfg.log_file)
logger.info(f"config={cfg}")

log_section(logger, "load and split corpus")
sents = load_sentences(cfg.corpus_name, lowercase=cfg.lowercase, keep_only_alpha=cfg.keep_only_alpha)
train, test = split_sentences(sents, train_ratio=cfg.train_ratio, seed=cfg.seed)
logger.info(f"sentences total={len(sents):,} train={len(train):,} test={len(test):,}")
logger.info(f"train_tokens={sum(len(s) for s in train):,} test_tokens={sum(len(s) for s in test):,}")

log_section(logger, "build vocabulary")
vocab = build_vocab(train, unk_min_count=cfg.unk_min_count)
logger.info(f"vocab_size={len(vocab)} unk_min_count={cfg.unk_min_count}")

log_section(logger, "fit n-gram models")
models: Dict[int, NgramLanguageModel] = {}
for n in cfg.ngram_orders:
    m = NgramLanguageModel(order=n, alpha=cfg.laplace_alpha)
    m.fit(train, vocab=vocab, logger=logger)
    models[n] = m

log_section(logger, "compute perplexity")
report: List[dict] = []
for n in cfg.ngram_orders:
    ppl, tokens, lp_sum = models[n].perplexity(test, logger=logger)
    report.append({"order": n, "perplexity": ppl, "tokens_scored": tokens, "log_prob_sum": lp_sum})

2026-07-05 14:19:53 | INFO  | part2_lm | config=Part2Config(corpus_name='gutenberg', seed=42, lowercase=True, keep_only_alpha=True, train_ratio=0.8, unk_min_count=1, ngram_orders=(1, 2, 3), laplace_alpha=1.0, num_samples=5, max_sample_length=40, log_file='part2_lm.log')


2026-07-05 14:19:53 | INFO  | part2_lm | --- load and split corpus ---


2026-07-05 14:19:55 | INFO  | part2_lm | sentences total=98,326 train=78,660 test=19,666


2026-07-05 14:19:55 | INFO  | part2_lm | train_tokens=1,709,588 test_tokens=426,481


2026-07-05 14:19:55 | INFO  | part2_lm | --- build vocabulary ---


...

2026-07-05 14:20:00 | INFO  | part2_lm | fit 3/3 order=3 vocab=24029 contexts=443415 unique_ngrams=1103966 elapsed_s=2.4759


2026-07-05 14:20:00 | INFO  | part2_lm | --- compute perplexity ---


2026-07-05 14:20:00 | INFO  | part2_lm | ppl 1/3 order=1 tokens_scored=446147 log_prob_sum=-2904933.5528 ppl=672.6048


2026-07-05 14:20:01 | INFO  | part2_lm | ppl 2/3 order=2 tokens_scored=446147 log_prob_sum=-3205664.3308 ppl=1319.7795


2026-07-05 14:20:01 | INFO  | part2_lm | ppl 3/3 order=3 tokens_scored=446147 log_prob_sum=-3976139.4742 ppl=7421.7749


## Q6 - Unigram and bigram MLE

MLE puts probability mass in proportion to observed counts. For the unigram
this is `P(w) = c(w) / N`; for the bigram it is
`P(w | w_prev) = c(w_prev, w) / c(w_prev)`. The two cells below show these raw,
unsmoothed numbers for a handful of words and one context.

In [10]:
unigram = models[1]
bigram = models[2]

top_words = [w for w, _ in Counter(t for s in train for t in s if t in vocab).most_common(5)]
uni_rows = [(w, unigram.prob((), w, smoothed=False)) for w in top_words]
print("Unigram MLE probabilities for the five most common training tokens:")
for w, p in uni_rows:
    print(f"  P({w!r}) = {p:.6f}")

context = ("the",)
bi_ctx_counts = bigram._ctx_counts.get(context, Counter())
bi_top = [w for w, _ in bi_ctx_counts.most_common(5)]
print()
print("Raw bigram MLE probabilities for the top-5 continuations of 'the':")
for w in bi_top:
    print(f"  P({w!r} | 'the') = {bigram.prob(context, w, smoothed=False):.6f}")

Unigram MLE probabilities for the five most common training tokens:


  P('the') = 0.059906
  P('and') = 0.042825
  P('of') = 0.031917
  P('to') = 0.021481
  P('a') = 0.015185

Raw bigram MLE probabilities for the top-5 continuations of 'the':
  P('lord' | 'the') = 0.053049
  P('<UNK>' | 'the') = 0.014506
  P('king' | 'the') = 0.014431
  P('children' | 'the') = 0.011463
  P('house' | 'the') = 0.011407


## Q7 - Add-1 (Laplace) smoothing

Raw MLE assigns probability zero to any bigram that never appeared in training.
Since only a tiny fraction of possible bigrams are actually observed, almost
every held-out sentence contains at least one unseen pair, which collapses its
probability to zero and drives perplexity to infinity.

Add-1 smoothing pretends we saw every possible bigram once and rescales the
counts:

`P_L(w | w_prev) = (c(w_prev, w) + 1) / (c(w_prev) + V)`

This redistributes a small mass to every unseen bigram, keeps log-probabilities
finite, and preserves the ranking of observed bigrams. The trade-off is that
mass is stolen from the observed events, and with a large vocabulary this
smoothing is heavy-handed.

The cell below shows the effect on the same context we just used, plus one
unseen continuation for comparison.

In [11]:
demo_words = bi_top + ["martian"]
print("P(w | 'the') under add-1 smoothing:")
for w in demo_words:
    p = bigram.prob(context, w, smoothed=True)
    seen = bi_ctx_counts.get(w, 0) > 0
    tag = "seen" if seen else "unseen"
    print(f"  P({w!r} | 'the') = {p:.6e}   ({tag})")

P(w | 'the') under add-1 smoothing:
  P('lord' | 'the') = 4.333770e-02   (seen)
  P('<UNK>' | 'the') = 1.185611e-02   (seen)
  P('king' | 'the') = 1.179511e-02   (seen)
  P('children' | 'the') = 9.370521e-03   (seen)
  P('house' | 'the') = 9.324774e-03   (seen)
  P('martian' | 'the') = 7.624508e-06   (unseen)


## Q8 - Perplexity on the held-out test set

Corpus perplexity uses the smoothed distribution: for the unigram this is
add-1 over a vocabulary of `V`, for the bigram it is add-1 over `V` per
context. Tokens scored per sentence are the real tokens plus `</s>`.

In [12]:
q8_df = pd.DataFrame([r for r in report if r["order"] in (1, 2)])
q8_df["model"] = q8_df["order"].map({1: "unigram", 2: "bigram (add-1)"})
q8_df = q8_df[["model", "order", "perplexity", "tokens_scored", "log_prob_sum"]]
save_dataframe(q8_df, ART_DIR / "perplexity_table.csv")
q8_df

,model,order,perplexity,tokens_scored,log_prob_sum
0,unigram,1,672.604760,446147,-2.904934e+06
1,bigram (add-1),2,1319.779509,446147,-3.205664e+06


## Q9 - Five ancestral samples from the bigram model

Sampling starts from `<s>` and draws the next word from the bigram
distribution until `</s>` is drawn or the length cap is reached. The samples
here use the observed MLE distribution rather than the smoothed one: with add-1
over a 24K vocabulary the smoothed distribution collapses to near-uniform and
produces word soup. Perplexity above still uses the smoothed distribution.

In [13]:
log_section(logger, "ancestral sampling from bigram")
rng = np.random.default_rng(cfg.seed)
samples = [bigram.sample(rng, max_length=cfg.max_sample_length) for _ in range(cfg.num_samples)]
sample_lines = [" ".join(s) for s in samples]
for i, line in enumerate(sample_lines, start=1):
    logger.info(f"sample {i}: {line}")
save_text([f"{i}. {line}" for i, line in enumerate(sample_lines, start=1)],
          ART_DIR / "sampled_sentences.txt")
for i, line in enumerate(sample_lines, start=1):
    print(f"{i}. {line}")

2026-07-05 14:20:01 | INFO  | part2_lm | --- ancestral sampling from bigram ---


2026-07-05 14:20:01 | INFO  | part2_lm | sample 1: whew


2026-07-05 14:20:01 | INFO  | part2_lm | sample 2: leonora tell me innocent loue me


2026-07-05 14:20:01 | INFO  | part2_lm | sample 3: but a roar to queer very kind


2026-07-05 14:20:01 | INFO  | part2_lm | sample 4: and courage for such an accompaniment


2026-07-05 14:20:01 | INFO  | part2_lm | sample 5: of the crofts to the prophetess the negligent falling back


1. whew
2. leonora tell me innocent loue me
3. but a roar to queer very kind
4. and courage for such an accompaniment
5. of the crofts to the prophetess the negligent falling back


## Q10 - Trigram and cross-model comparison

The trigram uses the same add-1 recipe over `V^2` contexts. Because most
trigrams are unseen and Laplace steals mass to cover them uniformly, the
trigram perplexity often ends up higher than the bigram (and, on our
vocabulary, higher than the unigram too). This is the sparsity vs smoothing
trade-off in its purest form.

The cell below rebuilds `perplexity_table.csv` to include all three orders and
saves a summary JSON that the report reads.

In [14]:
full_df = pd.DataFrame(report)
full_df["model"] = full_df["order"].map({1: "unigram", 2: "bigram (add-1)", 3: "trigram (add-1)"})
full_df = full_df[["model", "order", "perplexity", "tokens_scored", "log_prob_sum"]]
save_dataframe(full_df, ART_DIR / "perplexity_table.csv")

summary = {
    "config": {k: getattr(cfg, k) for k in cfg.__dataclass_fields__.keys()},
    "sentences": {"total": len(sents), "train": len(train), "test": len(test)},
    "tokens": {"train": sum(len(s) for s in train), "test": sum(len(s) for s in test)},
    "vocabulary_size": len(vocab),
    "perplexity": {row["model"]: row["perplexity"] for row in full_df.to_dict(orient="records")},
    "samples": sample_lines,
}
save_json(summary, ART_DIR / "model_summary.json")
full_df

,model,order,perplexity,tokens_scored,log_prob_sum
0,unigram,1,672.604760,446147,-2.904934e+06
1,bigram (add-1),2,1319.779509,446147,-3.205664e+06
2,trigram (add-1),3,7421.774870,446147,-3.976139e+06


## Persist to outputs/

In [15]:
for src_name, dst_name in [
    ("perplexity_table.csv", "part2_perplexity.csv"),
    ("sampled_sentences.txt", "part2_samples.txt"),
    ("model_summary.json", "part2_summary.json"),
]:
    (OUT_DIR / dst_name).write_bytes((ART_DIR / src_name).read_bytes())

logger.info("part 2 done - artifacts written to artifacts/part2_lm and outputs/")

2026-07-05 14:20:01 | INFO  | part2_lm | part 2 done - artifacts written to artifacts/part2_lm and outputs/
